In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# 1. 加载数据
print("正在加载数据...")
# 修改数据路径与RF保持一致
df_original = pd.read_excel('../../data/quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量 - 确认与RF模型相同的目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']  # 与RF模型保持一致

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 最优参数:
# n_estimators: 335
# max_depth: 14
# num_leaves: 64
# learning_rate: 0.10279735193954564
# subsample: 0.7533616827994455
# colsample_bytree: 0.792740064845325
# min_child_samples: 17
# 6. 创建并训练模型
print("\n开始训练LightGBM模型...")
# 优化的LightGBM参数
lgb_params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 64,                     # 从100改为64
    'max_depth': 14,                      # 从29改为14
    'learning_rate': 0.10279735193954564, # 从0.05改为0.10279735193954564
    'feature_fraction': 0.792740064845325, # 从0.9改为colsample_bytree值0.792740064845325
    'subsample': 0.7533616827994455,      # 从0.8改为0.7533616827994455
    'min_child_samples': 17,              # 从3改为17
    'subsample_freq': 5,                  # 保持不变
    'verbose': -1,                        # 保持不变
    'n_jobs': -1,                         # 保持不变
    'n_estimators': 335,                  # 从500改为335
    'random_state': 42                    # 保持不变
}

model = MultiOutputRegressor(lgb.LGBMRegressor(**lgb_params))

# 使用训练数据训练
model.fit(X_train_scaled, y_train)
print("模型训练完成！")

# 7. 在测试集上进行预测
y_pred = model.predict(X_test_scaled)

# 8. 评估模型性能
print("\n模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")